# Power the underpowered has_error=1 stratum at 7B

Sixth notebook. `05_7b_capability_check.ipynb` found the 7B gate result
`marginal` (59.0% accuracy vs. 50.0% baseline), and a stratified read found
two different problems: the `has_error=1` stratum's AUROC (0.761) point
estimate replicates 3B almost exactly (0.756), but with only 17 misgraded
items it is **underpowered** (registered minimum: 30). The `has_error=0`
(clean) stratum, by contrast, was already adequately powered (44 misgraded)
and its inversion (AUROC 0.280) is a confirmed result.

**This notebook fixes only the deficient side.** Drawing a fresh balanced
sample would re-cover the clean stratum, which already has enough data --
wasted GPU time. `pilot.data.load_fermat_extra_error_items` draws only more
`has_error=1` items, guaranteed disjoint from the 150 the reference run
already used (same seed, same ordering, structural not probabilistic).

**Sizing (worked out before running, not after):** observed grading-error
rate within `has_error=1` is 17/150 = 11.3%, 95% Wilson CI [7.2%, 17.4%].
`N_EXTRA = 200` clears the 90%-confidence target at the point-estimate rate
(160) with margin, and stays close even at the pessimistic end of the CI.
Fallback if the session runs long: `N_EXTRA = 115` (the point-estimate
expected-value target) -- flagged explicitly as more likely to still fall
short of 30, not a free substitute.

**Reuses `SCALEUP_PREREGISTRATION`'s existing 0.70 threshold** for the
has_error=1 stratum verdict once adequately powered, rather than inventing a
new bar for 7B -- that threshold was registered for exactly this question
(does entropy predict grading errors within the error-containing stratum)
and applies regardless of model size.

**Consistency check, not just a merge:** the reference run was full bf16, not
quantized. This session might land on a different GPU and fall back to
4-bit. If `QUANTIZED` doesn't match the reference run's, merging would
silently combine two different measurements -- the notebook checks this
explicitly and refuses to merge silently if it doesn't match.

**Run order:** cells 1-3 (install, auth, model -- same as 05), then 4
(extra-items sample), 5 (grading generation), 6 (merge + stratified
analysis), 7 (save).

In [1]:
# Install cell: GPU-dependent packages only.
# `datasets` is intentionally also in the local requirements.txt -- each
# environment installs its own copy independently, no conflict.
# torch is not installed explicitly: Colab GPU runtimes ship with it preinstalled.
# sentence-transformers is installed here, not in the optional NLI cell at
# the end, so pip resolves every version constraint ONCE before the model
# loads. Installing it mid-session could pull a different transformers
# version underneath an already-loaded model.
!pip install -q transformers accelerate bitsandbytes datasets qwen-vl-utils sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 58.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 64.9 MB/s eta 0:00:00:00:0100:01


In [2]:
# Auth & code/results access cell.
import json
import os
from getpass import getpass

from huggingface_hub import login

# --- Drive mount first: it holds both the model cache and the token store ---
from google.colab import drive

drive.mount("/content/drive")
PROJECT_DIR = "/content/drive/MyDrive/uncertainty-math-vlm"
DRIVE_MODEL_CACHE = f"{PROJECT_DIR}/model_cache"
os.makedirs(DRIVE_MODEL_CACHE, exist_ok=True)

# --- Tokens: entered ONCE, then cached on your Drive ---
# Deliberately not hardcoded in this notebook. This file is tracked in a
# public repo, and GitHub's secret scanning auto-revokes any ghp_ token that
# lands in a public commit -- so an inline token would stop working by
# itself. Drive is private to your account, survives runtime recycling, and
# git never touches it, so you get the same "no retyping" result safely.
TOKEN_FILE = f"{PROJECT_DIR}/.tokens.json"
RESET_TOKENS = False  # set True once to replace previously saved tokens


def get_token(name, prompt):
    """Return a saved token, prompting (once) and persisting it if absent."""
    tokens = {}
    if os.path.exists(TOKEN_FILE):
        with open(TOKEN_FILE) as f:
            tokens = json.load(f)
    if RESET_TOKENS or not tokens.get(name):
        tokens[name] = getpass(prompt).strip()
        with open(TOKEN_FILE, "w") as f:
            json.dump(tokens, f)
        os.chmod(TOKEN_FILE, 0o600)
        print(f"Saved {name} to Drive -- you will not be asked for it again.")
    return tokens[name]


HF_TOKEN = get_token("HF_TOKEN", "Hugging Face token (asked once): ")
GH_TOKEN = get_token("GH_TOKEN", "GitHub token with 'repo' scope (asked once): ")

if not HF_TOKEN.startswith("hf_"):
    raise ValueError(
        "Stored Hugging Face token does not start with 'hf_'. Set "
        "RESET_TOKENS = True and re-run this cell to replace it."
    )

login(token=HF_TOKEN)
print("Hugging Face login OK")

# --- Clone the repo (code + results live in the same repo for this pilot) ---
# Cloned anonymously: the repo is public, so read access needs no token, and
# keeping the token out of the clone URL means a clone error can never echo
# it into this notebook's saved output. The token is used only to push.
REPO_URL = "https://github.com/sepehrmaleki369/uncertainty-math-vlm.git"

# Remove any stale clone from a previous (possibly failed) run so this cell
# is safe to re-run -- git clone silently no-ops into a pre-existing
# directory, which would otherwise leave `repo/` incomplete without error.
!rm -rf repo
!git clone -q {REPO_URL} repo

# %pip (not !pip) installs into the *running kernel's* environment -- !pip
# can silently target a different Python install.
%pip install -q -e repo/

# An editable install writes an `__editable__.pilot-*.pth` file into
# site-packages, but .pth files are only processed by the `site` module at
# INTERPRETER STARTUP. The kernel is already running, so it never sees them
# and `import pilot` fails with ModuleNotFoundError even though the install
# reported success. Putting the repo on sys.path directly makes the package
# importable right now, with no kernel restart needed.
import importlib
import sys

REPO_DIR = os.path.abspath("repo")
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
importlib.invalidate_caches()

import pilot.data
import pilot.prompts
import pilot.parsing
import pilot.entropy

print(f"pilot package imported from: {os.path.dirname(pilot.__file__)}")

Mounted at /content/drive
Hugging Face login OK
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 4.8 MB/s eta 0:00:00
  Building editable for pilot (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
omegaconf 2.3.1 requires antlr4-python3-runtime==4.9.*, but you have antlr4-python3-runtime 4.11.0 which is incompatible.
pilot package imported from: /content/repo/pilot


In [3]:
# Model load cell. 7B by default -- this notebook exists specifically to test
# a model bigger than the 3B every other result in this project used.
#
# Falls back to a 4-bit quantized load if full precision does not fit, but
# says so loudly: a quantized model is a genuinely different measurement, not
# a transparent substitute, and silently reporting a quantized-model accuracy
# as if it were the same experiment would misrepresent the capability check.
import torch
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
QUANTIZED = False

try:
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        cache_dir=DRIVE_MODEL_CACHE,
    )
    print(f"Loaded {MODEL_ID} in bfloat16 (full precision).")
except torch.cuda.OutOfMemoryError:
    print(f"bfloat16 load of {MODEL_ID} did not fit -- falling back to 4-bit "
          "quantization. This changes what is being measured; the saved "
          "results record QUANTIZED=True so this is never silently glossed over.")
    from transformers import BitsAndBytesConfig

    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=quantization_config,
        device_map="auto",
        cache_dir=DRIVE_MODEL_CACHE,
    )
    QUANTIZED = True

processor = AutoProcessor.from_pretrained(MODEL_ID, cache_dir=DRIVE_MODEL_CACHE)

if torch.cuda.is_available():
    vram_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {torch.cuda.get_device_name(0)} ({vram_gib:.1f} GiB), "
          f"quantized={QUANTIZED}")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

Loaded Qwen/Qwen2.5-VL-7B-Instruct in bfloat16 (full precision).
GPU: NVIDIA A100-SXM4-40GB (39.5 GiB), quantized=False


In [4]:
# Extra-items sample cell. Draws ONLY new has_error=1 items, disjoint from the
# 150 the reference 7B run already used.
import logging

import pilot.data

logging.basicConfig(level=logging.WARNING, force=True)

SEED = 42
SKIP = 150          # has_error=1 items the reference run already covered
N_EXTRA = 200        # see the intro cell for the sizing rationale

extra_sample = pilot.data.load_fermat_extra_error_items(
    n_extra=N_EXTRA, seed=SEED, skip=SKIP
)
N_EXTRA = len(extra_sample)  # may shrink if the pool ran short -- keep in sync
print(f"{N_EXTRA} additional has_error=1 items drawn (items {SKIP} to {SKIP + N_EXTRA - 1} "
      f"in the seed={SEED} error-item ordering, disjoint from the reference run's first {SKIP})")

README.md:   0%|          | 0.00/3.74k [00:00<?, ?B/s]

data/train-00000-of-00010.parquet: reconstructing file:   0%|          |  0.00B /  467MB            

data/train-00000-of-00010.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00010.parquet: reconstructing file:   0%|          |  0.00B /  481MB            

data/train-00001-of-00010.parquet: downloading bytes:           |  0.00B            

...

data/train-00008-of-00010.parquet: downloading bytes:           |  0.00B            

data/train-00009-of-00010.parquet: reconstructing file:   0%|          |  0.00B /  493MB            

data/train-00009-of-00010.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2244 [00:00<?, ? examples/s]

200 additional has_error=1 items drawn (items 150 to 349 in the seed=42 error-item ordering, disjoint from the reference run's first 150)


In [5]:
# Grading generation for the extra items. Same K=5, same batch-backoff ladder
# as 05 -- identical generation code, just pointed at a different item set and
# a distinct checkpoint file.
import gc
import json
import os
import time

import torch
from qwen_vl_utils import process_vision_info
from tqdm.auto import tqdm

import pilot.prompts

K_GRADING = 5
TEMP = 0.7
_BATCH_LADDER = [5, 2, 1]
_batch_state = {"index": 0}

META_FIELDS = ("orig_q", "pert_a", "has_error", "handwriting_style", "image_quality")
INFRA_EXCEPTIONS = (ConnectionError, TimeoutError, torch.cuda.OutOfMemoryError, OSError)


def _generate_batch(messages, n: int, temperature: float):
    text_prompt = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text_prompt], images=image_inputs, videos=video_inputs,
        padding=True, return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs, max_new_tokens=512, do_sample=True,
            temperature=temperature, num_return_sequences=n,
        )

    trimmed = output_ids[:, inputs["input_ids"].shape[1]:]
    texts = processor.batch_decode(
        trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )
    del output_ids, inputs
    gc.collect()
    torch.cuda.empty_cache()
    return texts


def generate_grading(messages, n: int, temperature: float):
    texts = []
    last_exc = None
    while len(texts) < n:
        want = n - len(texts)
        size = min(_BATCH_LADDER[_batch_state["index"]], want)
        for attempt in range(3):
            try:
                texts += _generate_batch(messages, size, temperature)
                last_exc = None
                break
            except torch.cuda.OutOfMemoryError:
                gc.collect()
                torch.cuda.empty_cache()
                if _batch_state["index"] + 1 < len(_BATCH_LADDER):
                    _batch_state["index"] += 1
                    print(f"  OOM at batch {size}; dropping to "
                          f"{_BATCH_LADDER[_batch_state['index']]} for the rest of the run.",
                          flush=True)
                    size = min(_BATCH_LADDER[_batch_state["index"]], n - len(texts))
                    continue
                raise
            except INFRA_EXCEPTIONS as exc:
                last_exc = exc
                gc.collect()
                torch.cuda.empty_cache()
                if attempt < 2:
                    time.sleep(5)
        if last_exc is not None:
            raise last_exc
    return texts


CHECKPOINT_DIR = "/content/drive/MyDrive/uncertainty-math-vlm/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
model_slug = MODEL_ID.split("/")[-1]
# Distinct filename from 05's checkpoint: different item set entirely, and
# quantization mode is part of the key for the same reason as before -- a
# skip or sample under one mode is not evidence about the other.
extra_grading_path = (f"{CHECKPOINT_DIR}/grading_7b_extra_error_k{K_GRADING}_{model_slug}"
                      f"_n{N_EXTRA}_skip{SKIP}_seed{SEED}{'_4bit' if QUANTIZED else ''}.jsonl")

extra_grading_results = []
if os.path.exists(extra_grading_path):
    with open(extra_grading_path) as f:
        extra_grading_results = [json.loads(line) for line in f if line.strip()]
    valid = []
    for idx, entry in enumerate(extra_grading_results[:N_EXTRA]):
        item = extra_sample[idx]
        if not all(entry["item"].get(k) == item[k] for k in META_FIELDS):
            print(f"Checkpoint item {idx + 1} does not match sample order; resuming there.")
            break
        if len(entry.get("samples_raw", [])) != K_GRADING:
            break
        valid.append(entry)
    if len(valid) != len(extra_grading_results):
        with open(extra_grading_path, "w") as f:
            for e in valid:
                f.write(json.dumps(e, default=str) + "\n")
    extra_grading_results = valid
    print(f"Resuming from {len(extra_grading_results)} completed items")

if len(extra_grading_results) >= N_EXTRA:
    print(f"All {N_EXTRA} items already done.")
else:
    # Printed per item, not just tqdm's dynamic bar -- tqdm's carriage-return
    # redraw does not reliably render in every notebook viewer (confirmed
    # 2026-08-05: a run looked completely stalled with no way to tell whether
    # it was working or hung, and interrupting to check state cost real time).
    # A plain print() always appears in the cell's static output, so "no new
    # line in N minutes" becomes an unambiguous signal instead of a guess.
    print(f"Starting from item {len(extra_grading_results) + 1}/{N_EXTRA} "
          f"({N_EXTRA - len(extra_grading_results)} remaining)", flush=True)
    with tqdm(total=(N_EXTRA - len(extra_grading_results)) * K_GRADING,
              desc="grading (extra)", unit="sample") as pbar:
        for item_idx, item in enumerate(extra_sample):
            if item_idx < len(extra_grading_results):
                continue
            _t0 = time.time()
            messages = pilot.prompts.build_grading_messages(item["image"])
            texts = generate_grading(messages, K_GRADING, TEMP)
            _elapsed = time.time() - _t0
            entry = {
                "item": {k: item[k] for k in META_FIELDS},
                "samples_raw": texts,
                "quantized": QUANTIZED,
                "elapsed_seconds": _elapsed,
            }
            extra_grading_results.append(entry)
            with open(extra_grading_path, "a") as f:
                f.write(json.dumps(entry, default=str) + "\n")
                f.flush()
            pbar.update(K_GRADING)
            print(f"  item {item_idx + 1}/{N_EXTRA}: {_elapsed:.1f}s "
                  f"({len(extra_grading_results)}/{N_EXTRA} done)", flush=True)

print(f"extra_grading_results: {len(extra_grading_results)} items")

Resuming from 70 completed items
Starting from item 71/200 (130 remaining)


grading (extra):   0%|          | 0/650 [00:00<?, ?sample/s]

  item 71/200: 11.7s (71/200 done)
  item 72/200: 8.2s (72/200 done)
  item 73/200: 12.0s (73/200 done)
  item 74/200: 11.9s (74/200 done)
  item 75/200: 7.9s (75/200 done)
  item 76/200: 8.6s (76/200 done)
  item 77/200: 7.7s (77/200 done)
  item 78/200: 9.0s (78/200 done)
  item 79/200: 7.4s (79/200 done)
  item 80/200: 9.5s (80/200 done)
  item 81/200: 8.8s (81/200 done)
  item 82/200: 8.7s (82/200 done)
  item 83/200: 8.7s (83/200 done)
  item 84/200: 9.4s (84/200 done)
  item 85/200: 8.5s (85/200 done)
  item 86/200: 8.6s (86/200 done)
  item 87/200: 12.6s (87/200 done)
  item 88/200: 9.4s (88/200 done)
  item 89/200: 16.4s (89/200 done)
  item 90/200: 9.7s (90/200 done)
  item 91/200: 11.2s (91/200 done)
  item 92/200: 11.3s (92/200 done)
  item 93/200: 8.6s (93/200 done)
  item 94/200: 7.2s (94/200 done)
  item 95/200: 11.7s (95/200 done)
  item 96/200: 13.0s (96/200 done)
  item 97/200: 9.3s (97/200 done)
  item 98/200: 9.5s (98/200 done)
  item 99/200: 10.6s (99/200 done)
  it

In [6]:
# Merge with the reference run and re-run the stratified analysis.
#
# Reads the REFERENCE CHECKPOINT (raw text), not the already-scored CSV --
# the final CSV notebook 05 saved strips samples down to digit + entropy +
# correctness, which is why "why did the model miss these?" could not be
# answered from it. Manual inspection of the raw checkpoint (2026-08-05,
# before this fix) found real, concrete mechanisms: some wrong items are
# digit/reasoning self-contradictions (the reasoning text flags a problem,
# the digit still says 0 -- ~35-40% of digit=0 samples within the wrong
# items, after excluding "no error"-type false matches on the same keyword
# check, vs 11% overall at 3B; NOT yet a fully controlled comparison, since
# the 3B figure was not re-measured with the same false-positive filter).
# Other misses look like the model pattern-matching a solution's structure
# rather than independently re-deriving the arithmetic -- e.g. restating
# "19 x -18 x 23 = -7966" as confirmation without recomputing it (the true
# product is -7866). Both are testable at scale now that raw text is kept.
#
# The consistency check matters: merging a bf16 reference with a 4-bit extra
# batch (or vice versa) would silently average two different measurements
# together and report the result as one. This is the same class of mistake as
# notebook 04's cross-capture-mode checkpoint bug, one level up (across runs
# instead of within one run's checkpoint).
import importlib
import json
import os

import pandas as pd

import pilot.entropy
import pilot.parsing
import pilot.plotting

for m in (pilot.parsing, pilot.entropy, pilot.plotting):
    importlib.reload(m)

# Look for the reference checkpoint under EITHER precision suffix, not just
# the one matching this session's own QUANTIZED value. Computing the filename
# from this session's mode alone was a real bug caught in dry-run testing: if
# the reference run actually used the other precision, that guess resolves to
# a file that doesn't exist, and the notebook fails with a confusing "not
# found" instead of the intended, informative mismatch error below.
_ref_base = f"{CHECKPOINT_DIR}/grading_7b_k5_{model_slug}_n300_seed{SEED}"
_ref_candidates = {"bf16": f"{_ref_base}.jsonl", "4bit": f"{_ref_base}_4bit.jsonl"}
_found = {label: path for label, path in _ref_candidates.items() if os.path.exists(path)}

if not _found:
    raise AssertionError(
        f"No reference checkpoint found at either {_ref_candidates['bf16']} or "
        f"{_ref_candidates['4bit']}. It is the raw 2026-08-05 300-item run; "
        "without it there is nothing to merge the extra items into."
    )
if len(_found) > 1:
    raise AssertionError(
        f"Found reference checkpoints under BOTH precisions: {list(_found)}. "
        "That should never happen (only one 05 run has ever completed) -- "
        "resolve manually before merging, to avoid picking the wrong one."
    )
REFERENCE_CHECKPOINT = next(iter(_found.values()))

with open(REFERENCE_CHECKPOINT) as f:
    reference_entries = [json.loads(line) for line in f if line.strip()]
assert len(reference_entries) == 300, (
    f"Expected 300 reference items, found {len(reference_entries)} -- the "
    "checkpoint may be incomplete."
)

ref_quantized = bool(reference_entries[0]["quantized"])
if ref_quantized != QUANTIZED:
    raise RuntimeError(
        f"Reference run was quantized={ref_quantized}, this session is "
        f"quantized={QUANTIZED}. Refusing to merge two different measurements "
        "silently -- re-run this notebook on a GPU that matches the reference "
        "run's precision, or explicitly decide these are not comparable."
    )
print(f"Precision check OK: both runs quantized={QUANTIZED}")


def score_entry(entry):
    """Score one raw checkpoint entry, keeping the raw text and per-sample
    digits alongside the usual derived columns -- this is the whole point of
    reading the checkpoint instead of a stripped CSV."""
    digits = [pilot.parsing.parse_grading(t) for t in entry["samples_raw"]]
    labels = [None if d is None else str(d) for d in digits]
    majority, _ = pilot.entropy.majority_cluster(labels)
    said_error = majority == "1"
    return {
        "orig_q": entry["item"]["orig_q"],
        "pert_a": entry["item"]["pert_a"],
        "has_error": entry["item"]["has_error"],
        "grading_correct": majority in {"0", "1"} and said_error == bool(entry["item"]["has_error"]),
        "said_error": said_error,
        "reasoning_entropy": pilot.entropy.cluster_entropy(labels),
        "n_grading_parse_failures": sum(1 for d in digits if d is None),
        "majority_digit": majority,
        "parsed_digits": digits,               # e.g. [1, 0, 1, 1, 0] or [None, 1, ...]
        "all_grading_samples_raw": entry["samples_raw"],  # the full text, for inspection
        "model_id": MODEL_ID,
        "quantized": entry["quantized"],
    }


reference_df = pd.DataFrame(score_entry(e) for e in reference_entries)
extra_df = pd.DataFrame(score_entry(e) for e in extra_grading_results)

# Disjointness check, not just an assumption -- the whole design depends on it.
overlap = set(zip(reference_df["orig_q"], reference_df["pert_a"])) & \
          set(zip(extra_df["orig_q"], extra_df["pert_a"]))
assert not overlap, f"{len(overlap)} items overlap between reference and extra -- sampling bug"

combined = pd.concat([reference_df, extra_df], ignore_index=True)
print(f"Combined: {len(reference_df)} reference + {len(extra_df)} extra = {len(combined)} total")
print("Columns now include majority_digit, parsed_digits, and "
      "all_grading_samples_raw -- so wrong cases can be read, not just counted.")

gt = combined["has_error"].astype(bool)
print(f"  has_error=1: {int(gt.sum())} items, {int((~combined.loc[gt,'grading_correct']).sum())} misgraded")
print(f"  has_error=0: {int((~gt).sum())} items, {int((~combined.loc[~gt,'grading_correct']).sum())} misgraded")

print()
print("=" * 70)
print("STRATIFIED ANALYSIS (combined)")
print("=" * 70)
out = pilot.plotting.stratified_auroc(
    combined, "reasoning_entropy", "grading_correct", "has_error", n_boot=10000, seed=0
)
for level, s in out["strata"].items():
    minority = min(s["n_error"], s["n_correct"])
    powered = minority >= pilot.plotting.SCALEUP_PREREGISTRATION["min_minority_class"]
    print(f"  has_error={level}  n={s['n_items']:3d}  n_wrong={s['n_error']:3d}  "
          f"AUROC {s['auroc']:.3f} [{s['ci_low']:.3f}, {s['ci_high']:.3f}]  "
          f"minority={minority}  {'POWERED' if powered else 'still underpowered'}")
print(f"  sign_reversal      : {out['sign_reversal']}")
print(f"  pooled_understates : {out['pooled_understates']}")

print()
print("=" * 70)
print("VERDICT (reusing the registered 0.70 threshold, not a new one)")
print("=" * 70)
error_stratum = out["strata"][True]
minority = min(error_stratum["n_error"], error_stratum["n_correct"])
min_n = pilot.plotting.SCALEUP_PREREGISTRATION["min_minority_class"]
threshold = pilot.plotting.SCALEUP_PREREGISTRATION["reasoning_stratum_auroc_min"]

if minority < min_n:
    print(f"  Still underpowered ({minority} < {min_n}). N_EXTRA was not enough; "
          "draw more with load_fermat_extra_error_items(skip=350, ...) to extend further.")
elif error_stratum["auroc"] >= threshold and error_stratum["excludes_chance"]:
    print(f"  CONFIRMED: AUROC {error_stratum['auroc']:.3f} clears the registered "
          f"{threshold} threshold with adequate power ({minority} >= {min_n}).")
else:
    print(f"  NOT CONFIRMED: adequately powered ({minority} >= {min_n}) but AUROC "
          f"{error_stratum['auroc']:.3f} does not clear {threshold}, or its CI includes chance.")


Precision check OK: both runs quantized=False
Combined: 300 reference + 200 extra = 500 total
Columns now include majority_digit, parsed_digits, and all_grading_samples_raw -- so wrong cases can be read, not just counted.
  has_error=1: 350 items, 38 misgraded
  has_error=0: 150 items, 106 misgraded

STRATIFIED ANALYSIS (combined)
  has_error=False  n=150  n_wrong=106  AUROC 0.280 [0.200, 0.366]  minority=44  POWERED
  has_error=True  n=350  n_wrong= 38  AUROC 0.834 [0.768, 0.891]  minority=38  POWERED
  sign_reversal      : True
  pooled_understates : True

VERDICT (reusing the registered 0.70 threshold, not a new one)
  CONFIRMED: AUROC 0.834 clears the registered 0.7 threshold with adequate power (38 >= 30).


In [7]:
# Save cell: the combined CSV, Drive first then repo + push.
import subprocess
from datetime import datetime, timezone
from getpass import getpass

timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
csv_name = (f"grading_7b_stratum_powered_n{len(combined)}_"
            f"{'4bit_' if QUANTIZED else ''}{model_slug.lower()}_{timestamp}.csv")

drive_results = "/content/drive/MyDrive/uncertainty-math-vlm/results"
os.makedirs(drive_results, exist_ok=True)
combined.to_csv(f"{drive_results}/{csv_name}", index=False)
print(f"Backup written to {drive_results}/{csv_name}")

os.makedirs("repo/results", exist_ok=True)
csv_path = f"repo/results/{csv_name}"
combined.to_csv(csv_path, index=False)
print(f"Wrote {csv_path} ({len(combined)} rows)")

_REDACT = []


def git(*args):
    result = subprocess.run(["git", "-C", "repo", *args], capture_output=True, text=True)
    output = (result.stdout or "") + (result.stderr or "")
    for secret in _REDACT:
        if secret:
            output = output.replace(secret, "***")
    if result.returncode != 0 and output.strip():
        print(output.strip())
    return result


git("config", "user.email", "colab-pilot@localhost")
git("config", "user.name", "Colab Pilot Run")
git("add", f"results/{csv_name}")
commit = git("commit", "-m", f"Add 7B stratum-powered grading results: {csv_name}")
if commit.returncode != 0:
    print("git commit failed (see above) -- CSV is safe on Drive.")

GH_PUSH_TOKEN = (globals().get("GH_TOKEN") or "").strip()
if not GH_PUSH_TOKEN:
    GH_PUSH_TOKEN = getpass("GitHub token (to push results), then press Enter: ").strip()
_REDACT.append(GH_PUSH_TOKEN)

if not GH_PUSH_TOKEN:
    print("No token given -- skipping push. CSV is saved on Drive and in repo/results/.")
else:
    push_url = REPO_URL.replace("https://", f"https://{GH_PUSH_TOKEN}@")
    if git("fetch", push_url, "main").returncode == 0:
        if git("rebase", "FETCH_HEAD").returncode != 0:
            git("rebase", "--abort")
            print("Rebase onto remote failed; attempting push anyway.")
    if git("push", push_url, "HEAD:main").returncode == 0:
        print("Pushed results.")
    else:
        print("Push failed (see above). The CSV is safe on Drive and in "
              "repo/results/ -- retry the push without re-running the model.")

Backup written to /content/drive/MyDrive/uncertainty-math-vlm/results/grading_7b_stratum_powered_n500_qwen2.5-vl-7b-instruct_20260805T135021Z.csv
Wrote repo/results/grading_7b_stratum_powered_n500_qwen2.5-vl-7b-instruct_20260805T135021Z.csv (500 rows)
remote: Permission to sepehrmaleki369/uncertainty-math-vlm.git denied to sepehrmaleki369.
fatal: unable to access 'https://github.com/sepehrmaleki369/uncertainty-math-vlm.git/': The requested URL returned error: 403
Push failed (see above). The CSV is safe on Drive and in repo/results/ -- retry the push without re-running the model.
